# Mini-Project Sprint: Multi-Agent Travel Planner (Self-Contained)

**Prompt Engineering & Autonomous Reasoning**

This is the **all-in-one** version of the sprint. Everything the project needs — sample
data, utility helpers, local tools, worker agents, the critic, and the manager — is
implemented **directly inside this notebook**. There are no imports from other project
files, so you can run it top-to-bottom and present the whole system in one place.

The architecture:

- **Prompt contracts** turn a vague travel request into a structured object.
- **ReAct** gives each worker an inspectable `Thought → Action → Observation → Decision` trace.
- **Reflexion** lets a critic improve the first draft before the final answer.
- **Local tools** simulate external travel APIs so the demo stays deterministic and offline.

> Run mode: this notebook runs in **offline** mode by default — no API key, ₹0 cost, fully
> deterministic. A `live` OpenAI path is included and clearly marked so you can show how the
> same architecture would call a real model.

In [1]:
import json
import os
import re
from typing import Any, Dict, List, Optional


from openai import OpenAI




In [2]:
from dotenv import load_dotenv
load_dotenv()  # take environment variables from .env.

True

## Section 1 — Sample travel data (embedded)

In the original project these live in `data/*.json`. Here they are embedded as Python
objects so the notebook is standalone. The data is intentionally small and deterministic
so the source of every fact stays visible.

In [5]:
SAMPLE_FLIGHTS = [
    {
        "flight_id": "SQ-421", "origin": "Mumbai", "destination": "Singapore",
        "airline": "Singapore Airlines", "depart_time": "11:45", "arrive_time": "19:50",
        "duration_hours": 5.6, "stops": 0, "price_inr": 28500,
        "baggage": "25 kg checked + 7 kg cabin",
        "tags": ["direct", "daytime", "premium", "reliable"],
        "notes": "Comfortable direct daytime option; arrives before dinner.",
    },
    {
        "flight_id": "AI-342", "origin": "Mumbai", "destination": "Singapore",
        "airline": "Air India", "depart_time": "23:15", "arrive_time": "07:20",
        "duration_hours": 5.6, "stops": 0, "price_inr": 23600,
        "baggage": "25 kg checked + 7 kg cabin",
        "tags": ["direct", "red-eye", "budget"],
        "notes": "Cheaper direct flight but violates avoid-red-eye preference.",
    },
    {
        "flight_id": "6E-101", "origin": "Mumbai", "destination": "Singapore",
        "airline": "IndiGo", "depart_time": "06:10", "arrive_time": "14:05",
        "duration_hours": 5.4, "stops": 0, "price_inr": 21900,
        "baggage": "15 kg checked + 7 kg cabin",
        "tags": ["direct", "early-morning", "budget"],
        "notes": "Budget-friendly direct flight; early start may be tiring.",
    },
    {
        "flight_id": "UK-878", "origin": "Mumbai", "destination": "Singapore",
        "airline": "Vistara", "depart_time": "09:20", "arrive_time": "20:35",
        "duration_hours": 8.8, "stops": 1, "price_inr": 24800,
        "baggage": "20 kg checked + 7 kg cabin",
        "tags": ["one-stop", "daytime", "mid-budget"],
        "notes": "One-stop daytime route; slower than direct but avoids red-eye.",
    },
    {
        "flight_id": "SQ-403", "origin": "Delhi", "destination": "Singapore",
        "airline": "Singapore Airlines", "depart_time": "21:55", "arrive_time": "06:10",
        "duration_hours": 5.8, "stops": 0, "price_inr": 32700,
        "baggage": "25 kg checked + 7 kg cabin",
        "tags": ["direct", "red-eye", "premium"],
        "notes": "Strong airline choice but red-eye arrival.",
    },
    {
        "flight_id": "EK-503", "origin": "Mumbai", "destination": "Dubai",
        "airline": "Emirates", "depart_time": "19:20", "arrive_time": "21:15",
        "duration_hours": 3.4, "stops": 0, "price_inr": 21500,
        "baggage": "25 kg checked + 7 kg cabin",
        "tags": ["direct", "evening", "premium"],
        "notes": "Direct evening route to Dubai.",
    },
]

In [6]:
SAMPLE_HOTELS = [
    {
        "hotel_id": "SG-H01", "destination": "Singapore",
        "name": "Harbour View City Hotel", "area": "Tanjong Pagar",
        "price_per_night_inr": 11200, "rating": 4.3,
        "amenities": ["breakfast", "metro nearby", "family rooms"],
        "tags": ["mid-budget", "city", "food", "metro"],
        "notes": "Good access to Chinatown, Maxwell Food Centre, and MRT.",
    },
    {
        "hotel_id": "SG-H02", "destination": "Singapore",
        "name": "Orchard Urban Stay", "area": "Orchard",
        "price_per_night_inr": 15600, "rating": 4.5,
        "amenities": ["breakfast", "pool", "shopping district"],
        "tags": ["premium", "shopping", "central"],
        "notes": "Convenient but may exceed a strict mid-budget hotel cap.",
    },
    {
        "hotel_id": "SG-H03", "destination": "Singapore",
        "name": "Bugis Heritage Inn", "area": "Bugis",
        "price_per_night_inr": 8900, "rating": 4.1,
        "amenities": ["metro nearby", "self check-in"],
        "tags": ["budget", "culture", "metro", "walkable"],
        "notes": "Strong budget option near Arab Street and Kampong Glam.",
    },
    {
        "hotel_id": "SG-H04", "destination": "Singapore",
        "name": "Marina Bay Skyline Hotel", "area": "Marina Bay",
        "price_per_night_inr": 23800, "rating": 4.7,
        "amenities": ["skyline view", "pool", "breakfast"],
        "tags": ["views", "premium", "landmark"],
        "notes": "Excellent views, but likely outside a mid-budget plan.",
    },
    {
        "hotel_id": "DXB-H01", "destination": "Dubai",
        "name": "Creekside Business Hotel", "area": "Deira",
        "price_per_night_inr": 7900, "rating": 4.0,
        "amenities": ["metro nearby", "breakfast"],
        "tags": ["budget", "metro", "business"],
        "notes": "Affordable and convenient for old Dubai.",
    },
]

In [8]:
SAMPLE_ACTIVITIES = [
    {
        "activity_id": "SG-A01", "destination": "Singapore",
        "name": "Chinatown and Maxwell Food Centre walk", "area": "Chinatown",
        "duration_hours": 3, "best_time": "Evening", "cost_inr": 1600,
        "tags": ["food", "culture", "walkable", "low-cost"],
        "notes": "Works well on arrival day if the flight lands before evening.",
    },
    {
        "activity_id": "SG-A02", "destination": "Singapore",
        "name": "Gardens by the Bay + Marina Bay walk", "area": "Marina Bay",
        "duration_hours": 5, "best_time": "Afternoon to night", "cost_inr": 2800,
        "tags": ["views", "landmark", "nature", "photo"],
        "notes": "Pairs well with sunset and the evening light show.",
    },
    {
        "activity_id": "SG-A03", "destination": "Singapore",
        "name": "National Gallery and Civic District", "area": "Civic District",
        "duration_hours": 4, "best_time": "Morning", "cost_inr": 1200,
        "tags": ["culture", "museum", "history", "indoor"],
        "notes": "Good choice for culture-focused travellers.",
    },
    {
        "activity_id": "SG-A04", "destination": "Singapore",
        "name": "Kampong Glam food and heritage trail", "area": "Bugis",
        "duration_hours": 3, "best_time": "Late afternoon", "cost_inr": 1500,
        "tags": ["food", "culture", "walkable", "low-cost"],
        "notes": "Fits well with a Bugis hotel base.",
    },
    {
        "activity_id": "SG-A05", "destination": "Singapore",
        "name": "Sentosa half-day: beaches and cable car", "area": "Sentosa",
        "duration_hours": 6, "best_time": "Morning to afternoon", "cost_inr": 5200,
        "tags": ["leisure", "views", "family", "premium"],
        "notes": "Good optional add-on but can stretch the budget.",
    },
    {
        "activity_id": "SG-A06", "destination": "Singapore",
        "name": "Hawker centre tasting route", "area": "Multiple",
        "duration_hours": 4, "best_time": "Lunch or dinner", "cost_inr": 2200,
        "tags": ["food", "local", "low-cost"],
        "notes": "Strong fit for food-focused travellers.",
    },
    {
        "activity_id": "DXB-A01", "destination": "Dubai",
        "name": "Old Dubai souks and creek abra ride", "area": "Deira/Bur Dubai",
        "duration_hours": 4, "best_time": "Evening", "cost_inr": 1800,
        "tags": ["culture", "food", "low-cost"],
        "notes": "Good low-cost culture block for Dubai.",
    },
]


In [9]:
def load_travel_data() -> Dict[str, List[Dict[str, Any]]]:
    """Load sample travel data for flights, hotels, and activities."""
    return {
        "flights": SAMPLE_FLIGHTS,
        "hotels": SAMPLE_HOTELS,
        "activities": SAMPLE_ACTIVITIES,
    }

travel_data = load_travel_data()
print(f"Flights: {len(travel_data['flights'])}, Hotels: {len(travel_data['hotels'])}, Activities: {len(travel_data['activities'])}")

Flights: 6, Hotels: 5, Activities: 7


## Section 2 — Utility helpers

`pretty_json` for readable output, `parse_json_safely` to survive malformed model JSON,
and `load_project_env` to report the runtime mode.

In [15]:

def load_project_env() -> Dict[str, str]:
    """Report runtime settings. Defaults to offline so the notebook needs no API key."""
    return {
        "demo_mode": os.getenv("DEMO_MODE", "live").strip().lower(),
        "model_name": os.getenv("OPENAI_MODEL", "gpt-4o-mini").strip(),
        "temperature": os.getenv("OPENAI_TEMPERATURE", "0.2").strip(),
        "langsmith_tracing": os.getenv("LANGSMITH_TRACING", "false").strip().lower(),
    }

def parse_json_safely(prompt: str, system: str, temperature: float = 0.0) -> dict:
    """LLM call that must return JSON. Strips markdown fences and parses."""
    raw = llm(prompt, system=system + "\nRespond ONLY with valid JSON. No preamble, no markdown fences.",
              temperature=temperature)
    cleaned = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.MULTILINE).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        # one repair attempt — a common production trick
        repaired = llm(f"Fix this into strict valid JSON, output only the JSON:\n{raw}",
                       system="You repair malformed JSON.", temperature=0.0)
        repaired = re.sub(r"^```(?:json)?|```$", "", repaired.strip(), flags=re.MULTILINE).strip()
        return json.loads(repaired)



def pretty_json(data: Any) -> str:
    """Format Python data as readable JSON text."""
    return json.dumps(data, indent=2, ensure_ascii=False)



In [11]:
settings = load_project_env()

In [16]:
pretty_json(settings)


'{\n  "demo_mode": "live",\n  "model_name": "gpt-4o-mini",\n  "temperature": "0.2",\n  "langsmith_tracing": "false"\n}'

## Section 3 — The LLM client

A thin wrapper around OpenAI JSON-mode calls. **Offline mode never calls it** — every agent
below has a deterministic offline path. The class is included so you can show exactly where
a real model would plug in.

In [19]:
class LLMClient:
    """Small wrapper around OpenAI JSON-mode calls (only used in live mode)."""
    def __init__(self, demo_mode: str = "offline", model_name: str = "gpt-4o-mini", temperature: float = 0.2):
        self.demo_mode = demo_mode
        self.model_name = model_name
        self.temperature = temperature
        # self._client = None


        if self.demo_mode == "live":
            self._client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


    def complete_json(self, system_prompt: str, user_payload: Dict[str, Any], purpose: str) -> Dict[str, Any]:
        """Call the model and ask for a json object"""

        if self.demo_mode != 'live':
            raise RuntimeError("LLMClient.complete_json called in offline mode. This is a demo-only function.")

        try:
            response = self._client.completions.create(
                model=self.model_name,
                temperature=self.temperature,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": json.dumps(user_payload, indent=2, ensure_ascii=False)},
                ],
            )
        except Exception as exc:
            raise RuntimeError(f"LLMClient.complete_json failed while handling {purpose}: {exc}") from exc

        # The response is expected to be a JSON object
        content = response.choices[0].message.content or "{}"
        parsed = parse_json_safely(content)
        return parsed
        



In [20]:
def _normalise(text: Any) -> str:
    """Normalise text for comparison: lowercase, strip, collapse whitespace."""
    return str(text or "").strip().lower()


def _score_tags(item_tags: List[str], desired_tags: List[str]) -> int:
    """Score overlap between item tags and desired tags."""
    item_tag_set = {_normalise(tag) for tag in item_tags}
    desired_tag_set = {_normalise(tag) for tag in desired_tags}
    return len(item_tag_set.intersection(desired_tag_set))

    

In [21]:
def search_flights(requirements: Dict[str, Any], flights: List[Dict[str, Any]], top_k: int =3) -> List[Dict[str, Any]]:
    """Search for flights that match the requirements and return top_k results."""
    origin = _normalise(requirements.get("origin"))
    destination = _normalise(requirements.get("destination"))
    avoid_red_eye = bool(requirements.get("avoid_red_eye", False))

    candidates: List[Dict[str, Any]] = []
    for flight in flights:
        if origin and _normalise(flight.get("origin")) != origin:
            continue
        if destination and _normalise(flight.get("destination")) != destination:
            continue

        score = 0
        tags = [_normalise(tag) for tag in flight.get("tags", [])]

        if 'direct' in tags:
            score += 4
        if 'daytime' in tags:
            score += 3
        if 'budget' in tags or 'mid-budget' in tags:
            score += 2
        if avoid_red_eye and 'red-eye' in tags:
            score -= 8  # heavy penalty for red-eye if to be avoided

        score -= int(flight.get('stops', 0))
        score =- int(flight.get("price_inr",0))/20000 ### this 20000 is a scaling factor to keep the score in a reasonable range and to get this number user average flight price in India to Singapore is around 20000 INR. This is a rough estimate and can be adjusted based on actual data.

        enriched = dict(flight)
        enriched["selection_score"] = score
        candidates.append(enriched)

    return sorted(candidates, key=lambda x: x["selection_score"], reverse=True)[:top_k]



In [22]:
def search_hotels(requirements: Dict[str, Any], hotels: List[Dict[str, Any]], top_k: int = 3) -> List[Dict[str, Any]]:
    """Search local hotel options for the requested destination."""
    destination = _normalise(requirements.get("destination"))
    duration_days = int(requirements.get("duration_days") or 4)
    nights = max(duration_days - 1, 1)
    hotel_budget = int(requirements.get("hotel_budget_inr") or 0)
    preferences = requirements.get("preferences", [])

    candidates: List[Dict[str, Any]] = []
    for hotel in hotels:
        if destination and _normalise(hotel.get("destination")) != destination:
            continue

        total_price = int(hotel.get("price_per_night_inr", 0)) * nights
        score = 0
        score += _score_tags(hotel.get("tags", []), preferences) * 3
        score += int(float(hotel.get("rating", 0)) * 2)
        if hotel_budget and total_price <= hotel_budget:
            score += 5
        elif hotel_budget:
            score -= 5
        if "metro nearby" in [_normalise(a) for a in hotel.get("amenities", [])]:
            score += 2

        enriched = dict(hotel)
        enriched["nights"] = nights
        enriched["total_price_inr"] = total_price
        enriched["selection_score"] = score
        candidates.append(enriched)

    return sorted(candidates, key=lambda item: (item["selection_score"], -item["total_price_inr"]), reverse=True)[:top_k]


In [23]:
def search_activities(requirements: Dict[str, Any], activities: List[Dict[str, Any]], top_k: int = 6) -> List[Dict[str, Any]]:
    """Search local activities aligned with destination and preferences."""
    destination = _normalise(requirements.get("destination"))
    preferences = requirements.get("preferences", [])

    candidates: List[Dict[str, Any]] = []
    for activity in activities:
        if destination and _normalise(activity.get("destination")) != destination:
            continue

        score = _score_tags(activity.get("tags", []), preferences) * 4
        if "low-cost" in [_normalise(tag) for tag in activity.get("tags", [])]:
            score += 1

        enriched = dict(activity)
        enriched["selection_score"] = score
        candidates.append(enriched)

    return sorted(candidates, key=lambda item: (item["selection_score"], -item["cost_inr"]), reverse=True)[:top_k]


In [24]:
def estimate_trip_budget(
    selected_flight: Dict[str, Any],
    selected_hotel: Dict[str, Any],
    selected_activities: List[Dict[str, Any]],
    traveller_count: int,
    ) -> Dict[str, Any]:
    """Estimate the total trip budget based on selected options and traveller count."""
    flight_total = int(selected_flight.get("price_inr", 0) * traveller_count)
    hotel_total = int(selected_hotel.get("total_price_inr", 0) * traveller_count)
    activities_total = sum(int(act.get("cost_inr", 0)) for act in selected_activities) * traveller_count
    return {
        "flight_inr": flight_total,
        "hotel_inr": hotel_total,
        "activities_inr": activities_total,
        "total_trip_inr": flight_total + hotel_total + activities_total,
    }

In [25]:
_demo_reqs = {"origin": "Mumbai", "destination": "Singapore", "avoid_red_eye": True,
              "duration_days": 4, "hotel_budget_inr": 45000, "preferences": ["food", "culture", "views"]}

print("Top flight:", search_flights(_demo_reqs, SAMPLE_FLIGHTS)[0]["flight_id"])
print("Top hotel: ", search_hotels(_demo_reqs, SAMPLE_HOTELS)[0]["name"])



Top flight: 6E-101
Top hotel:  Bugis Heritage Inn


### Worker agents (ReAct traces)

Each worker calls a tool, then returns an auditable `Thought → Action → Observation → Decision`
trace alongside the ranked options and a single recommendation.

In [27]:
class FlightWorker:
    """Worker that searches and explain flight options"""
    def __init__(self, llm: LLMClient, flights: List[Dict[str, Any]]):
        self.llm = llm
        self.flights = flights

    def run(self, requirements: Dict[str, Any]) -> Dict[str, Any]:
        options = search_flights(requirements, self.flights)
        if not options:
            return {
                "worker": "flight_search",
                "react_trace": [
                    "Thought: Need route-matching flights.",
                    "Action: search_flights",
                    "Observation: No matching local records found.",
                    "Decision: Ask the manager to clarify origin/destination or expand data.",
                ],
                "options": [], "recommendation": None,
                "rationale": "No matching flights were available in the local dataset.",
            }

        if self.llm.demo_mode == "live":
            system_prompt = (
                "You are a flight-search worker in a travel-planning agent team. "
                "Return JSON with keys: worker, react_trace, options, recommendation, rationale. "
                "Be concise and choose one best option. Do not invent flights."
            )
            return self.llm.complete_json(
                system_prompt, {"requirements": requirements, "flight_options": options},
                purpose="flight worker ranking",
            )

        best = options[0]
        return {
            "worker": "flight_search",
            "react_trace": [
                "Thought: Need flights that match route and hard constraints.",
                "Action: search_flights(origin, destination, avoid_red_eye)",
                f"Observation: Found {len(options)} candidate option(s).",
                f"Decision: Recommend {best['flight_id']} because it best balances constraints and cost.",
            ],
            "options": options, "recommendation": best,
            "rationale": best.get("notes", "Best-ranked option from local flight search."),
        }


            

    

In [29]:
class HotelWorker:
    """Worker that searches and explains hotel options."""
    def __init__(self, llm: LLMClient, hotels: List[Dict[str, Any]]):
        self.llm = llm
        self.hotels = hotels

    def run(self, requirements: Dict[str, Any]) -> Dict[str, Any]:
        options = search_hotels(requirements, self.hotels)
        if not options:
            return {
                "worker": "hotel_search",
                "react_trace": [
                    "Thought: Need destination-matching hotels.",
                    "Action: search_hotels",
                    "Observation: No matching local records found.",
                    "Decision: Ask the manager to clarify destination or expand data.",
                ],
                "options": [], "recommendation": None,
                "rationale": "No matching hotels were available in the local dataset.",
            }

        if self.llm.demo_mode == "live":
            system_prompt = (
                "You are a hotel-search worker in a travel-planning agent team. "
                "Return JSON with keys: worker, react_trace, options, recommendation, rationale. "
                "Choose a hotel that respects budget and preferences. Do not invent hotels."
            )
            return self.llm.complete_json(
                system_prompt, {"requirements": requirements, "hotel_options": options},
                purpose="hotel worker ranking",
            )


        #### Only run when demo mode is ofline

        best = options[0]
        budget_phrase = "within budget" if budget and best["total_price_inr"] <= budget else "best trade-off available"
        return {
            "worker": "hotel_search",
            "react_trace": [
                "Thought: Need hotel options matching destination, budget, and preferences.",
                "Action: search_hotels(destination, hotel_budget, preferences)",
                f"Observation: Found {len(options)} candidate option(s).",
                f"Decision: Recommend {best['name']} as the {budget_phrase}.",
            ],
            "options": options, "recommendation": best,
            "rationale": best.get("notes", "Best-ranked option from local hotel search."),
        }


        

In [30]:
class ActivityWorker:
    """Worker that searches and groups activities for the itinerary."""

    def __init__(self, llm: LLMClient, activities: List[Dict[str, Any]]):
        self.llm = llm
        self.activities = activities

    def run(self, requirements: Dict[str, Any]) -> Dict[str, Any]:
        options = search_activities(requirements, self.activities)
        if not options:
            return {
                "worker": "activity_planner",
                "react_trace": [
                    "Thought: Need destination-matching activities.",
                    "Action: search_activities",
                    "Observation: No matching local records found.",
                    "Decision: Ask the manager to clarify destination or expand data.",
                ],
                "options": [], "recommendation": [],
                "rationale": "No matching activities were available in the local dataset.",
            }

        if self.llm.demo_mode == "live":
            system_prompt = (
                "You are an activity-planning worker in a travel-planning agent team. "
                "Return JSON with keys: worker, react_trace, options, recommendation, rationale. "
                "Recommend a balanced shortlist only from provided activities."
            )
            return self.llm.complete_json(
                system_prompt, {"requirements": requirements, "activity_options": options},
                purpose="activity worker shortlisting",
            )

        selected = options[: min(len(options), 5)]
        return {
            "worker": "activity_planner",
            "react_trace": [
                "Thought: Need activities that match trip preferences and fit across days.",
                "Action: search_activities(destination, preferences)",
                f"Observation: Found {len(options)} candidate activity option(s).",
                "Decision: Shortlist the strongest matches and leave room for pacing.",
            ],
            "options": options, "recommendation": selected,
            "rationale": "Selected activities maximise overlap with stated preferences while keeping variety.",
        }


### The Reflexion critic

The critic inspects the first draft for budget breaches, red-eye violations, day-count
mismatches, and missing preferences, then returns issues, revision instructions, and a
quality score.

In [31]:
class ItineraryCritic:
    """Critic that evaluates an itinerary and proposes refinements."""
    def __init__(self, llm: LLMClient):
        self.llm = llm
    
    def evaluate(
        self,
        requirements: Dict[str, Any],
        draft_itinerary: Dict[str, Any],
        worker_outputs: Dict[str, Dict[str, Any]],
    ) -> Dict[str, Any]:

        if self.llm.demo_mode == "live":
            system_prompt = (
                "You are a Reflexion critic for a travel-planning agent team. "
                "Return JSON with keys: overall_assessment, issues, revision_instructions, quality_score. "
                "Check budget, pacing, constraints, unsupported assumptions, and preference fit. "
                "Do not invent facts beyond the provided worker outputs."

            )
            return self.llm.complete_json(
                system_prompt,
                {"requirements": requirements, "draft_itinerary": draft_itinerary, "worker_outputs": worker_outputs},
                purpose="itinerary critique",
            )




### The manager agent

The manager owns orchestration: it extracts requirements (a prompt contract), delegates to
workers, synthesises a draft, runs the critic, and applies the refinement.

In [37]:
class TravelPlannerManager:
    """Manager-worker travel planner with Reflexion refinement."""
    def __init__(self, llm: LLMClient, travel_data: Dict[str, List[Dict[str, Any]]] | None = None):
        self.llm = llm
        self.travel_data = travel_data or load_travel_data()
        self.flight_worker = FlightWorker(llm, self.travel_data["flights"])
        self.hotel_worker = HotelWorker(llm, self.travel_data["hotels"])
        self.activity_worker = ActivityWorker(llm, self.travel_data["activities"])
        self.critic = ItineraryCritic(llm)

    def parse_request(self, user_request: str) -> Dict[str, Any]:
        """Extract structured requirements from a free-text travel request."""
        if self.llm.demo_mode == "live":
            print("live model is enabled")
            system_prompt = (
                "You are the manager agent for a travel-planning team. "
                "Extract requirements from a user request. Return JSON with keys: "
                "origin, destination, duration_days, traveller_count, budget_level, "
                "hotel_budget_inr, avoid_red_eye, preferences, assumptions, missing_information. "
                "Use null for unknown values and do not invent precise budgets unless stated."
            )
            return self.llm.complete_json(
                system_prompt, {"user_request": user_request},
                purpose="manager requirement extraction",
            )
        

    def run_workers(self, requirements: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
        """Run all specialist workers."""
        return {
            "flight": self.flight_worker.run(requirements),
            "hotel": self.hotel_worker.run(requirements),
            "activities": self.activity_worker.run(requirements),
        }

    def create_draft_itinerary(
        self, requirements: Dict[str, Any], worker_outputs: Dict[str, Dict[str, Any]],
    ) -> Dict[str, Any]:
        """Create the first itinerary draft from worker outputs."""
        if self.llm.demo_mode == "live":
            system_prompt = (
                "You are the synthesis manager in a travel-planning team. "
                "Create a first-draft itinerary using only the worker outputs. "
                "Return JSON with keys: selected_flight, selected_hotel, daily_plan, budget, tradeoffs, assumptions. "
                "Keep the plan practical and day-wise."
            )
            return self.llm.complete_json(
                system_prompt, {"requirements": requirements, "worker_outputs": worker_outputs},
                purpose="initial itinerary synthesis",
            )

    def refine_itinerary(
        self, requirements: Dict[str, Any], draft_itinerary: Dict[str, Any],
        critique: Dict[str, Any], worker_outputs: Dict[str, Dict[str, Any]],
    ) -> Dict[str, Any]:
        """Apply critic feedback and produce a final itinerary."""
        if self.llm.demo_mode == "live":
            system_prompt = (
                "You are the final manager agent. Refine the itinerary using critic feedback. "
                "Return JSON with keys: final_itinerary, changes_after_critique, final_notes. "
                "Keep recommendations grounded in the worker outputs."
            )
            return self.llm.complete_json(
                system_prompt,
                {"requirements": requirements, "draft_itinerary": draft_itinerary,
                 "critique": critique, "worker_outputs": worker_outputs},
                purpose="final itinerary refinement",
            )

    


    def run(self, user_request: str) -> Dict[str, Any]:
        """Run the complete manager-worker-reflexion pipeline."""
        requirements = self.parse_request(user_request)
        worker_outputs = self.run_workers(requirements)
        draft_itinerary = self.create_draft_itinerary(requirements, worker_outputs)
        critique = self.critic.evaluate(requirements, draft_itinerary, worker_outputs)
        final_output = self.refine_itinerary(requirements, draft_itinerary, critique, worker_outputs)
        return {
            "user_request": user_request,
            "requirements": requirements,
            "worker_outputs": worker_outputs,
            "draft_itinerary": draft_itinerary,
            "critique": critique,
            "final_output": final_output,
        }


    

In [38]:
settings

{'demo_mode': 'live',
 'model_name': 'gpt-4o-mini',
 'temperature': '0.2',
 'langsmith_tracing': 'false'}

In [34]:
llm = LLMClient(
    demo_mode=settings["demo_mode"],
    model_name=settings["model_name"],
    temperature=float(settings["temperature"]),
)

In [39]:
llm

In [40]:
travel_data

{'flights': [{'flight_id': 'SQ-421',
   'origin': 'Mumbai',
   'destination': 'Singapore',
   'airline': 'Singapore Airlines',
   'depart_time': '11:45',
   'arrive_time': '19:50',
   'duration_hours': 5.6,
   'stops': 0,
   'price_inr': 28500,
   'baggage': '25 kg checked + 7 kg cabin',
   'tags': ['direct', 'daytime', 'premium', 'reliable'],
   'notes': 'Comfortable direct daytime option; arrives before dinner.'},
  {'flight_id': 'AI-342',
   'origin': 'Mumbai',
   'destination': 'Singapore',
   'airline': 'Air India',
   'depart_time': '23:15',
   'arrive_time': '07:20',
   'duration_hours': 5.6,
   'stops': 0,
   'price_inr': 23600,
   'baggage': '25 kg checked + 7 kg cabin',
   'tags': ['direct', 'red-eye', 'budget'],
   'notes': 'Cheaper direct flight but violates avoid-red-eye preference.'},
  {'flight_id': '6E-101',
   'origin': 'Mumbai',
   'destination': 'Singapore',
   'airline': 'IndiGo',
   'depart_time': '06:10',
   'arrive_time': '14:05',
   'duration_hours': 5.4,
   'st

In [35]:

manager = TravelPlannerManager(llm=llm, travel_data=travel_data)

In [41]:
user_request = (
    "Plan a 4-day trip from Mumbai to Singapore for two people. Keep it mid-budget, "
    "avoid red-eye flights, prefer food, city views, and cultural sites. "
    "Keep the total hotel budget under ₹45,000."
)


requirements = manager.parse_request(user_request)

RuntimeError: LLMClient.complete_json failed while handling manager requirement extraction: Missing required arguments; Expected either ('model' and 'prompt') or ('model', 'prompt' and 'stream') arguments to be given